# From hot days to a daily footprint

This worked example uses fabricated data. Two reference summers give every cell a P90 of 30 °C. Hand labelled answers are included. The example illustrates what the project library will produce; it can be run before the detection and footprint modules are implemented.

In [ ]:
import csv
from collections import defaultdict
from math import isclose, radians, sin
from pathlib import Path

from scipy.io import netcdf_file

fixtures = Path("data/examples/synthetic")
if not fixtures.is_dir():
    fixtures = Path("../data/examples/synthetic")

with (fixtures / "worked_example.csv").open(encoding="utf-8", newline="") as sample:
    rows = list(csv.DictReader(sample))

with netcdf_file(fixtures / "events_fixture.nc", mmap=False) as events:
    flags = events.variables["heatwave_flag"][:]
    assert flags.shape == (92, 2, 3)
    assert int((flags == 1).sum()) == 9
    assert int((flags == -1).sum()) == 1

daily = defaultdict(list)
for row in rows:
    daily[row["date"]].append(row)

assert [sum(int(row["heatwave_flag"]) == 1 for row in group)
        for group in daily.values()] == [3, 3, 3, 0, 0]

The two western cells in the southern row share an edge and form component 1. The eastern cell in the northern row touches that group diagonally and forms component 2. The unknown cell stays unknown. The example uses the nominal edges of the 0.1° grid to convert the cell counts to areas.

In [ ]:
def area_km2(latitude):
    radius = 6371.0088
    north = radians(latitude + 0.05)
    south = radians(latitude - 0.05)
    return radius**2 * (sin(north) - sin(south)) * radians(0.1)


june_3 = daily["2022-06-03"]
components = defaultdict(list)
for row in june_3:
    label = int(row["footprint_id"])
    if label > 0:
        components[label].append(row)

assert [len(components[key]) for key in (1, 2)] == [2, 1]
assert [(row["lat"], row["lon"]) for row in components[1]] == [("45.4", "9.1"), ("45.4", "9.2")]
assert [(row["lat"], row["lon"]) for row in components[2]] == [("45.5", "9.3")]

total = 0.0
for label, members in components.items():
    area = sum(area_km2(float(row["lat"])) for row in members)
    peak = max(float(row["tx_c"]) - float(row["p90_c"]) for row in members)
    assert peak == {1: 4.0, 2: 3.0}[label]
    total += area
    print(f"June 3 · component {label} · {len(members)} cells · {area:.2f} km² · {peak:.1f} °C peak exceedance")

direct_area = sum(area_km2(float(row["lat"])) for row in june_3 if row["heatwave_flag"] == "1")
assert isclose(total, direct_area)
print(f"Total affected area: {total:.2f} km²")

The complete case notebook will use the modules' outputs to plot daily affected area, a selected daily footprint map, and the temperature and threshold at one cell.